# GenAI, LLM Comparison, and Responsible AI

Owner: Member 6

This notebook completes the final project module by comparing the 01-05 verification pipeline with GenAI/LLM alternatives and by evaluating the system through responsible AI risks, limitations, scalability, and future work.

## Role in the Project

The first five modules build an interpretable misinformation verification pipeline: claim extraction, dataset EDA, entity linking, KG reasoning, and Bayesian final verdicts. This module asks whether that pipeline is responsible enough for a real-world misinformation setting and how it compares with a GenAI-only or RAG-based approach.

No live LLM call is made in this notebook. That is a deliberate responsible AI choice: the comparison should be reproducible and grounded in the project evidence, not in a one-off generated answer.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'LIAR_dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODULE_DIR = PROJECT_ROOT / '06_GenAI_Responsible_AI'
sys.path.insert(0, str(MODULE_DIR))

from run_responsible_ai_analysis import run_responsible_ai_analysis

SUMMARY_PATH = MODULE_DIR / 'responsible_ai_summary.json'
ANALYSIS_PATH = MODULE_DIR / 'responsible_ai_analysis.md'

PROJECT_ROOT, SUMMARY_PATH, ANALYSIS_PATH

(PosixPath('/Users/shanusharma/AIaP-Group_Project'),
 PosixPath('/Users/shanusharma/AIaP-Group_Project/06_GenAI_Responsible_AI/responsible_ai_summary.json'),
 PosixPath('/Users/shanusharma/AIaP-Group_Project/06_GenAI_Responsible_AI/responsible_ai_analysis.md'))

## Project Evidence Used

The comparison below uses concrete outputs from the earlier notebooks: the LIAR EDA summary, extraction handoff, entity-linking summary, KG reasoning summary, and Bayesian final verdict summary.

In [2]:
results = run_responsible_ai_analysis(PROJECT_ROOT)
metrics = results['metrics']
metrics

{'liar_total_rows': 12791,
 'liar_label_count': 6,
 'claim_extraction_records': 500,
 'mean_extraction_confidence': 0.7735,
 'unknown_subject_type_count': 211,
 'entity_link_coverage_percent': 54.8,
 'property_mapping_percent': 0.2,
 'average_linking_confidence': 0.374,
 'kg_unknown_percent': 99.8,
 'kg_supported_count': 1,
 'bayesian_uncertain_percent': 99.8,
 'bayesian_likely_true_count': 1,
 'average_decision_confidence': 0.039}

## Evidence From Notebooks 01-05

A high-quality responsible AI analysis should not be generic. The table below links each project stage to the evidence it produced and explains what that evidence means for responsible deployment.

In [3]:
results['module_evidence']

[{'module': '01 Claim Extraction',
  'method': 'RoBERTa NER plus spaCy dependency parsing',
  'project_evidence': '500 triples; mean extraction confidence 0.7735; 211 UNKNOWN subjects',
  'responsible_ai_interpretation': 'The system creates auditable subject-relation-object evidence, but extraction confidence is not a truth score and parser errors can flow downstream.'},
 {'module': '02 Dataset EDA',
  'method': 'LIAR dataset profiling and preprocessing review',
  'project_evidence': '12791 rows across 6 truth labels; 26 duplicate statements; 3566 missing speaker jobs',
  'responsible_ai_interpretation': 'The political dataset is useful for benchmarking but carries selection, speaker, topic, and label bias risks.'},
 {'module': '03 Entity Linking and KR',
  'method': 'Wikidata-compatible entity and property representation',
  'project_evidence': '274 of 500 records have any entity link; property mapping coverage is 0.2%; average linking confidence is 0.374',
  'responsible_ai_interpret

## Key Critical Finding

The pipeline is transparent but deliberately low-coverage. Entity linking and KG reasoning are the bottlenecks: many claims cannot be safely mapped to a precise KG fact, so the Bayesian module mostly returns `uncertain`. This is not a failure of the responsible AI design; it is an honest signal that the available evidence is insufficient.

In [4]:
{
    'entity_link_coverage_percent': metrics['entity_link_coverage_percent'],
    'property_mapping_percent': metrics['property_mapping_percent'],
    'kg_unknown_percent': metrics['kg_unknown_percent'],
    'bayesian_uncertain_percent': metrics['bayesian_uncertain_percent'],
    'average_decision_confidence': metrics['average_decision_confidence'],
}

{'entity_link_coverage_percent': 54.8,
 'property_mapping_percent': 0.2,
 'kg_unknown_percent': 99.8,
 'bayesian_uncertain_percent': 99.8,
 'average_decision_confidence': 0.039}

## GenAI/LLM Comparison

A GenAI-only fact-checker would likely provide an answer for nearly every claim. That can look stronger in a demo, but in misinformation verification the ability to answer is not the same as the ability to verify. The table compares the current pipeline, an unconstrained LLM-only approach, and a safer RAG hybrid.

In [5]:
results['comparison_rows']

[{'dimension': 'Evidence traceability',
  'current_pipeline': 'High: triples, entity IDs, KG status, evidence text, and Bayesian reasoning are stored.',
  'llm_only': 'Often weak unless explicitly forced to cite sources; generated rationales may sound plausible without evidence.',
  'rag_hybrid': 'Potentially high if retrieved passages, KG IDs, prompts, and model outputs are logged together.'},
 {'dimension': 'Coverage',
  'current_pipeline': 'Low-to-moderate: strict entity linking and KG rules leave most claims unknown.',
  'llm_only': 'High surface coverage because an LLM can answer almost every claim.',
  'rag_hybrid': 'Moderate-to-high if retrieval expands evidence while abstention remains allowed.'},
 {'dimension': 'Factuality risk',
  'current_pipeline': 'Lower overclaiming risk because the system abstains when KG evidence is missing.',
  'llm_only': 'Higher hallucination and overconfidence risk, especially on political or time-sensitive claims.',
  'rag_hybrid': 'Lower than LLM-

## Why LLM-Only Is Not Enough

An LLM-only verifier has useful language abilities: it can paraphrase claims, identify missing context, and produce readable explanations. However, it also introduces serious risks for AT3's real-world AI framing: hallucinated evidence, prompt sensitivity, weak reproducibility, training-data bias, and overconfident language. In this project, the correct role for GenAI is not to replace the pipeline, but to support evidence retrieval and explanation under strict constraints.

## Responsible AI Risk Register

The following register connects each risk to evidence from this project, the possible harm, current controls, and recommended improvements.

In [6]:
results['risk_register']

[{'risk_area': 'Hallucinated or unsupported evidence',
  'project_evidence': 'KG reasoning returned 99.8% unknown results.',
  'potential_harm': 'A system that still gives definitive answers could falsely accuse or falsely clear claims.',
  'current_control': 'The KG and Bayesian modules abstain when evidence is weak.',
  'recommended_improvement': 'Use RAG with citation verification and require source-level evidence before final labels.',
  'severity': 'High'},
 {'risk_area': 'False positives',
  'project_evidence': 'Bayesian output kept 99.8% of claims uncertain.',
  'potential_harm': 'True or nuanced claims could be labelled misinformation, harming public trust.',
  'current_control': 'Conservative 0.65 threshold and explicit uncertainty class.',
  'recommended_improvement': 'Route low-evidence or high-impact claims to human review.',
  'severity': 'High'},
 {'risk_area': 'False negatives',
  'project_evidence': 'The current KG layer found no contradicted claims in the 500-record ha

## False Positives and False Negatives

False positives occur when true or nuanced claims are labelled misinformation. False negatives occur when false claims are not challenged. In the current project, the larger practical issue is not aggressive false positives because the Bayesian module mostly abstains. The larger coverage issue is that false claims may remain `uncertain` because KG evidence is missing. This is safer than calling them true, but it is not enough for a production fact-checking system.

In [7]:
final_summary = json.loads((PROJECT_ROOT / '05_Bayesian_Inference' / 'final_verdict_summary.json').read_text())
final_summary.get('reference_bucket_alignment')

{'available_claims': 500,
 'alignment_rate': 0.392,
 'note': 'Reference labels are used for evaluation only, not to compute final probabilities.',
 'confusion': [{'reference_bucket': 'likely false',
   'final_verdict': 'uncertain',
   'count': 129},
  {'reference_bucket': 'likely true',
   'final_verdict': 'likely true',
   'count': 1},
  {'reference_bucket': 'likely true',
   'final_verdict': 'uncertain',
   'count': 175},
  {'reference_bucket': 'uncertain',
   'final_verdict': 'uncertain',
   'count': 195}]}

## Bias and Fairness Reflection

LIAR is a political dataset with concentrated speakers, topics, parties, and contexts. Speaker-history counts can be useful metadata, but they can also punish or favour speakers based on historical coverage rather than the current claim. The Bayesian module therefore uses speaker history only as a weak prior and keeps the original LIAR label separate as `reference_label` for evaluation rather than evidence.

In [8]:
dataset_summary = json.loads((PROJECT_ROOT / '02_Problem_Dataset_EDA' / 'dataset_summary.json').read_text())
{
    'top_speakers': dataset_summary['top_speakers'][:5],
    'top_subjects': dataset_summary['top_subjects'][:5],
    'missing_values': {
        'speaker_job': dataset_summary['missing_values']['speaker_job'],
        'state': dataset_summary['missing_values']['state'],
        'context': dataset_summary['missing_values']['context'],
    },
}

{'top_speakers': [{'speaker': 'barack-obama', 'count': 611},
  {'speaker': 'donald-trump', 'count': 343},
  {'speaker': 'hillary-clinton', 'count': 297},
  {'speaker': 'mitt-romney', 'count': 212},
  {'speaker': 'john-mccain', 'count': 189}],
 'top_subjects': [{'subject': 'economy', 'count': 1432},
  {'subject': 'health-care', 'count': 1426},
  {'subject': 'taxes', 'count': 1218},
  {'subject': 'federal-budget', 'count': 937},
  {'subject': 'education', 'count': 926}],
 'missing_values': {'speaker_job': 3566, 'state': 2748, 'context': 131}}

## Scalability and Deployment

The current pipeline can scale technically, but responsible deployment requires more than speed. The extraction and Bayesian stages can run in batches. Entity linking and KG querying need caching, rate-limit handling, and better property coverage. LLM-only verification would be easy to prototype but harder to govern at scale because cost, reproducibility, hallucination risk, and prompt variation all become operational problems.

## Recommended Future Work

The strongest future direction is a hybrid system: use LLMs for claim decomposition, query generation, and readable summaries, but require retrieved sources, KG evidence, calibrated uncertainty, and human review for high-impact claims.

In [9]:
results['future_work']

[{'priority': 1,
  'improvement': 'Replace single-triple extraction with multi-claim decomposition and coreference resolution.',
  'why_it_matters': 'Political claims often contain multiple factual units and pronouns that break simple triples.',
  'expected_effect': 'Higher recall and cleaner downstream KG queries.'},
 {'priority': 2,
  'improvement': 'Expand entity linking with candidate ranking, aliases, and confidence calibration.',
  'why_it_matters': 'Entity-link coverage is the main bottleneck before KG reasoning.',
  'expected_effect': 'More claims reach evidence-based support or contradiction decisions.'},
 {'priority': 3,
  'improvement': 'Add live Wikidata/DBpedia/SPARQL property checks plus trusted web retrieval.',
  'why_it_matters': 'The current KG layer mostly reasons over local descriptions and simple class rules.',
  'expected_effect': 'Better factual coverage while keeping evidence traceable.'},
 {'priority': 4,
  'improvement': 'Use an LLM only inside a retrieval-grou

## Report Handoff

The runner writes a polished markdown analysis and report-ready CSV tables. These files can be used directly in the final AT3 report sections on theoretical justification, critical reflection, responsible AI, scalability, and future work.

In [10]:
results['artifacts']

{'module_evidence_table': 'report_assets/tables/responsible_ai_module_evidence.csv',
 'comparison_table': 'report_assets/tables/genai_pipeline_comparison.csv',
 'risk_register': 'report_assets/tables/responsible_ai_risk_register.csv',
 'future_work_table': 'report_assets/tables/responsible_ai_future_work.csv',
 'summary_json': '06_GenAI_Responsible_AI/responsible_ai_summary.json',
 'analysis_markdown': '06_GenAI_Responsible_AI/responsible_ai_analysis.md'}

## Final Position

The responsible conclusion is that this project should not claim to be a complete automated fact-checker. It is better framed as an interpretable verification support system. Compared with a GenAI-only approach, it is more auditable and less prone to unsupported confident answers. Compared with a mature RAG system, it has lower evidence coverage. The best next version would combine this pipeline's traceability with retrieval-grounded LLM support and human oversight.